# Cadre mathématique de la détection de régimes latents pour les liquidity providers en DeFi

### Une approche probabiliste du market-making dynamique sur Uniswap V3

---

## 1. Introduction : formalisation du problème

Considérons un market-maker automatisé (AMM) tel qu'Uniswap V3, où un fournisseur de liquidité (FL) peut concentrer son capital dans une plage de prix personnalisée. La rentabilité et le risque de cette stratégie dépendent de manière critique du *comportement* du marché — le prix évolue-t-il lentement, permettant l'accumulation de frais, ou est-il volatile, exposant le FL à des pertes impermanentes (PI) ?

Ce comportement de marché n'est **pas directement observable**. Je postule qu'il est gouverné par un ensemble fini de **régimes latents (cachés)**, notés $Z_t \in \{1,2,\dots,K\}$. Dans mon application, j'interprète $K=3$ régimes comme suit :
- $Z_t = 1$ (Goldilocks) : faible volatilité, capture élevée de frais — idéal pour la liquidité concentrée.
- $Z_t = 2$ (Tendanciel) : mouvements directionnels avec volatilité modérée.
- $Z_t = 3$ (Toxique) : forte volatilité, risque de sélection adverse — dangereux pour les FL passifs.

J'observe cependant un vecteur de variables de marché à chaque pas horaire $t$ :
$$
\mathbf{X}_t = \big[ \text{log\_rendement}_t,\; \text{nombre\_trades}_t,\; \text{frais\_totaux\_usd}_t,\; \text{vol\_réalisée}_t \big]^T.
$$

Mon objectif est d'inférer la séquence de régimes cachés la plus probable $Z_{1:T}$ étant donné la séquence observée $\mathbf{X}_{1:T}$. Il s'agit d'un problème classique d'**inférence probabiliste dans un système dynamique**, qui sera abordé à l'aide d'un Modèle de Markov Caché (MMC).

## 2. Le modèle de Markov caché : un modèle de probabilité jointe

Un MMC est défini par deux processus stochastiques :

### 2.1 La chaîne de Markov latente (dynamique des régimes)
La séquence d'états cachés $\{Z_t\}$ suit une chaîne de Markov du premier ordre avec une matrice de transition homogène dans le temps $\mathbf{A} = (A_{ij})_{K\times K}$ :
$$
P(Z_t = j \mid Z_{t-1} = i, Z_{t-2}, \dots) = P(Z_t = j \mid Z_{t-1} = i) = A_{ij},
$$
avec $\sum_{j=1}^K A_{ij}=1$ pour chaque ligne $i$. La distribution initiale des états est $\boldsymbol{\pi} = (\pi_1,\dots,\pi_K)$ où $\pi_i = P(Z_1 = i)$.

### 2.2 Le processus d'émission (observations)
Étant donné l'état caché courant $Z_t$, l'observation $\mathbf{X}_t$ est indépendante de toutes les observations et états passés et futurs :
$$
P(\mathbf{X}_t \mid Z_t = k, \mathbf{X}_{1:t-1}, Z_{1:t-1}) = P(\mathbf{X}_t \mid Z_t = k).
$$

### 2.3 La distribution jointe
Ces deux hypothèses me permettent de factoriser la probabilité jointe complète des états et des observations :
$$
P(Z_{1:T}, \mathbf{X}_{1:T}) = P(Z_1) \prod_{t=2}^{T} P(Z_t \mid Z_{t-1}) \prod_{t=1}^{T} P(\mathbf{X}_t \mid Z_t).
$$
Cette factorisation élégante est le fondement de tous les algorithmes d'inférence pour les MMC, tels que l'algorithme avant–arrière et le décodage de Viterbi.

## 3. L'hypothèse d'émission gaussienne : un choix de modélisation

Pour rendre le modèle tractable, je suppose que la distribution d'émission pour chaque état est une loi normale multivariée avec une matrice de covariance diagonale (c'est-à-dire que les variables sont conditionnellement indépendantes étant donné l'état) :
$$
\mathbf{X}_t \mid Z_t = k \;\sim\; \mathcal{N}(\boldsymbol{\mu}_k,\; \boldsymbol{\Sigma}_k), \qquad \boldsymbol{\Sigma}_k = \operatorname{diag}(\sigma_{k,1}^2,\dots,\sigma_{k,d}^2).
$$

### 3.1 Estimation des paramètres (étape M)
Sous cette hypothèse gaussienne, les estimateurs du maximum de vraisemblance des paramètres, étant données les probabilités d'état postérieures $\gamma_t(k) = P(Z_t = k \mid \mathbf{X}_{1:T})$, sont les moments empiriques pondérés :
$$
\hat{\boldsymbol{\mu}}_k = \frac{\sum_{t=1}^T \gamma_t(k)\,\mathbf{x}_t}{\sum_{t=1}^T \gamma_t(k)}, \qquad
\hat{\sigma}_{k,j}^2 = \frac{\sum_{t=1}^T \gamma_t(k)\,(x_{t,j} - \hat{\mu}_{k,j})^2}{\sum_{t=1}^T \gamma_t(k)}.
$$

### 3.2 Sélection du modèle (BIC)
Pour choisir le nombre d'états cachés $K$, j'utilise le Critère d'Information Bayésien (BIC), qui équilibre l'adéquation du modèle et sa complexité :
$$
\text{BIC} = -2\ln\hat{L} + m\ln T,
$$
où $\hat{L}$ est la vraisemblance maximisée du modèle et $m$ est le nombre de paramètres libres. Pour mon MMC à covariance diagonale, $m = K(K-1) + (K-1) + 2Kd$ (paramètres de transition, initiaux et d'émission). L'hypothèse gaussienne est cruciale ici car elle définit la vraisemblance $\hat{L}$ et le nombre de paramètres.

## 4. Vérification des hypothèses fondamentales 

Avant de faire confiance aux inférences du modèle, je dois vérifier ses hypothèses sous-jacentes. Le tableau ci-dessous résume chaque hypothèse, son implication mathématique pour le MMC, et les tests statistiques que j'ai effectués (détaillés dans `00_stats.ipynb`).

| **Hypothèse** | **Implication mathématique** | **Méthode** |
| :--- | :--- | :--- |
| **Stationnarité** | Pour que la matrice de transition $\mathbf{A}$ soit constante dans le temps, le processus générateur de données doit être (tendanciellement) stationnaire. Des variables non stationnaires (par ex., une tendance dans `nombre_trades`) impliqueraient que la dynamique des régimes elle-même dérive. | **Tests ADF et KPSS :** J'ai appliqué ces tests complémentaires de racine unitaire et de stationnarité à chaque série de variables. Une interprétation conjointe (rejet ADF + non-rejet KPSS) soutient la stationnarité. |
| **Émissions Gaussiennes** | Les formules de l'étape M ci-dessus sont des estimateurs du maximum de vraisemblance uniquement sous normalité. Si la vraie distribution a des queues épaisses (comme pour `log_rendement`), la vraisemblance est mal spécifiée et le BIC devient une heuristique, non un sélecteur de modèle exact. | **Test de Jarque–Bera et graphiques Q-Q :** J'ai testé la normalité de chaque variable, à la fois globalement et au sein de partitions d'état heuristiques. Les graphiques Q-Q ont révélé visuellement des déviations (par ex., queues épaisses). |
| **Indépendance Conditionnelle (Covariance Diagonale)** | Supposer $\boldsymbol{\Sigma}_k = \operatorname{diag}(\dots)$ signifie que, à l'intérieur d'un état, les variables sont non corrélées. Si c'est faux, un modèle diagonal perd de l'information et peut nécessiter davantage d'états pour compenser, risquant le surapprentissage. | **Test de sphéricité de Bartlett :** Pour chaque état heuristique, j'ai testé $H_0$ : matrice de corrélation = identité. Un rejet suggère qu'un modèle à covariance complète capturerait mieux la structure intra-état. |
| **Propriété de Markov du Premier Ordre** | Le modèle suppose que $Z_t$ dépend uniquement de $Z_{t-1}$, c'est-à-dire $Z_t \perp Z_{t-2} \mid Z_{t-1}$. Si une mémoire plus longue existe (par ex., $Z_t$ dépend de $Z_{t-2}$), la matrice $\mathbf{A}$ estimée est une moyenne mal spécifiée de dynamiques d'ordre supérieur. | **Test du rapport de vraisemblance de Billingsley :** En utilisant la séquence d'états heuristique, j'ai testé les chaînes de Markov du premier ordre contre celles du second ordre. Une petite valeur de $p$ indique le rejet de l'hypothèse du premier ordre. |

Ces tests me guident dans l'interprétation des limites du modèle et des raffinements potentiels (par ex., utiliser une covariance complète, ou même des émissions de Student‑$t$).

## 5. Du modèle à la stratégie : interprétation des résultats

Après avoir estimé les paramètres du modèle via l'algorithme de Baum–Welch (une instance de l'Espérance-Maximisation), je peux retrouver la séquence d'états cachés la plus probable à l'aide de l'**algorithme de Viterbi** :
$$
\hat{Z}_{1:T} = \arg\max_{Z_{1:T}} P(Z_{1:T} \mid \mathbf{X}_{1:T}; \hat{\boldsymbol{\theta}}),
$$
où $\hat{\boldsymbol{\theta}}$ désigne les paramètres estimés $\{\boldsymbol{\pi}, \mathbf{A}, \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k\}$.

Chaque état inféré correspond à une condition de marché distincte, caractérisée par son vecteur moyen d'émission $\boldsymbol{\mu}_k$ :
- **Goldilocks** (faible `vol_réalisée`, élevé `frais_totaux_usd`) : idéal pour la liquidité concentrée. Un FL devrait **réduire sa plage** pour maximiser la capture de frais.
- **Tendanciel** (volatilité modérée `vol_réalisée`, `log_rendement` directionnel) : un environnement équilibré ; une plage équilibrée peut être appropriée.
- **Toxique** (forte `vol_réalisée`, faible efficacité des frais) : la sélection adverse domine. Le FL devrait **élargir la plage** ou se retirer temporairement pour atténuer les pertes impermanentes et le LVR.

Ainsi, le MMC transforme un flux de données brutes on-chain en un **signal de risque probabiliste** qui guide directement une stratégie dynamique de fourniture de liquidité.

## 6. Conclusion

J'ai formalisé le problème de détection des régimes de marché latents pour un pool Uniswap V3 à l'aide d'un Modèle de Markov Caché. En spécifiant les variables aléatoires, la structure du modèle et les hypothèses sous-jacentes, j'ai construit un cadre mathématique rigoureux qui connecte la théorie classique des probabilités à une application DeFi moderne. Les hypothèses ont été examinées de manière critique à travers des tests statistiques, et les états décodés finaux fournissent des recommandations actionnables pour les fournisseurs de liquidité.

Ce travail démontre comment les modèles probabilistes de séries temporelles peuvent être mis à profit pour améliorer la prise de décision en finance décentralisée — une illustration parfaite de la synergie entre la modélisation mathématique et la finance quantitative qui est au cœur du curriculum de master en processus stochastiques et ingénierie financière. 
                
<p align="center">Master auquel je souhaite, avec votre accord, postuler.</p>